# Topic 2 — Ridge, Lasso & ElasticNet (Regularization)

> **Goal of this notebook:** understand *why* plain linear regression overfits when there are many (or correlated) features, and how **regularization** — Ridge (L2), Lasso (L1) and ElasticNet (L1+L2) — fixes it. We'll see it in action on a deliberately high-dimensional dataset.

**Contents**
1. Concept & intuition
2. The mathematics (L2, L1, ElasticNet penalties)
3. When to use each
4. The dataset: Diabetes + polynomial features
5. Exploratory data analysis
6. Baseline: where plain linear regression breaks down
7. Training & evaluating Ridge, Lasso, ElasticNet
8. Regularization paths & choosing alpha by cross-validation
9. Ridge from scratch (closed form)
10. Pros, cons & when to use
11. Summary

## 1. Concept & Intuition

Plain linear regression minimises only the prediction error (MSE). When you have **many features**, **correlated features**, or **few samples**, it can fit the training noise — the learned weights blow up and the model **overfits**, performing poorly on new data.

**Regularization** adds a penalty on the size of the weights to the cost function. The model is now rewarded for being accurate *and* for keeping weights small. This **shrinks** coefficients toward zero, trading a little training accuracy for much better generalisation.

Three flavours:
- **Ridge (L2):** shrinks all weights smoothly toward zero, but rarely exactly zero. Great for correlated features.
- **Lasso (L1):** can drive weights *exactly* to zero — performing automatic **feature selection**.
- **ElasticNet:** a mix of both — feature selection *and* stable handling of correlated groups.

## 2. The Mathematics

Let the ordinary least-squares cost be $\text{MSE} = \frac{1}{m}\sum_i (\hat y^{(i)} - y^{(i)})^2$. Each method adds a penalty controlled by $\alpha$ (larger $\alpha$ = stronger regularization).

**Ridge (L2 penalty):**
$$J = \text{MSE} + \alpha \sum_{j=1}^{n} w_j^2$$

**Lasso (L1 penalty):**
$$J = \text{MSE} + \alpha \sum_{j=1}^{n} |w_j|$$

**ElasticNet (both):**
$$J = \text{MSE} + \alpha \left( r \sum_j |w_j| + (1-r) \sum_j w_j^2 \right)$$

where $r$ (the `l1_ratio`) sets the mix between L1 and L2.

**Why does L1 zero out weights but L2 doesn't?** The L1 penalty $|w|$ has a constant-magnitude gradient that keeps pushing small weights all the way to 0, producing a **sparse** solution. The L2 penalty $w^2$ has a gradient $2w$ that vanishes as $w \to 0$, so it shrinks but never fully removes a feature.

> **Always scale your features before regularizing** — the penalty acts on weight magnitudes, so features must be on a comparable scale or the penalty is applied unfairly.

## 3. When to Use Each

| Situation | Best choice |
|-----------|-------------|
| Many correlated features, keep all | **Ridge** |
| Many features, suspect only a few matter | **Lasso** (feature selection) |
| Many correlated features *and* want selection | **ElasticNet** |
| Want a simple, interpretable subset of features | **Lasso** |

`alpha = 0` reduces all three back to ordinary linear regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV)
from sklearn.metrics import r2_score, mean_squared_error

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Dataset: Diabetes + Polynomial Features

We use the built-in **Diabetes** dataset (442 patients, 10 baseline features) where the target is disease progression one year later. With only 10 features it's too easy for regularization to matter — so we deliberately expand it with **degree-2 polynomial features** (squares and pairwise products). This creates ~65 features, many of them correlated, which is exactly the regime where plain linear regression overfits and regularization shines.

In [ ]:
data = load_diabetes(as_frame=True)
X_raw = data.data
y = data.target

print('Original shape:', X_raw.shape)
print('Features:', list(X_raw.columns))
print('Target: disease progression after one year')
X_raw.head()

In [ ]:
# Expand to degree-2 polynomial features (squares + interactions)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_raw)
feature_names = poly.get_feature_names_out(X_raw.columns)

print('Expanded feature count:', X_poly.shape[1])
print('Samples:', X_poly.shape[0])
print('-> many features relative to samples = overfitting risk')

## 5. Exploratory Data Analysis

Let's confirm the expanded features are highly correlated with each other (multicollinearity) — the condition that destabilises plain linear regression.

In [ ]:
corr = np.corrcoef(X_poly, rowvar=False)
# fraction of feature pairs with |correlation| > 0.8 (excluding the diagonal)
mask = ~np.eye(corr.shape[0], dtype=bool)
high = np.abs(corr[mask]) > 0.8
print(f'Highly correlated feature pairs (|r|>0.8): {high.sum()//2}')
print(f'Max off-diagonal correlation: {np.abs(corr[mask]).max():.3f}')

In [ ]:
# Split, then scale (fit scaler on train only)
X_train, X_test, y_train, y_test = train_test_split(
    X_poly, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
y_train = np.asarray(y_train)
y_test = np.asarray(y_test)
print('Train:', X_train_s.shape, ' Test:', X_test_s.shape)

## 6. Baseline: Where Plain Linear Regression Breaks Down

Watch the gap between training and test performance — a classic overfitting signature.

In [ ]:
lr = LinearRegression().fit(X_train_s, y_train)

print('Plain Linear Regression')
print(f'  Train R2: {r2_score(y_train, lr.predict(X_train_s)):.4f}')
print(f'  Test  R2: {r2_score(y_test, lr.predict(X_test_s)):.4f}')
print(f'  Largest |coefficient|: {np.abs(lr.coef_).max():.1f}')

The train R² is far higher than the test R², and some coefficients are huge — the model memorised noise. Now let's regularize.

## 7. Training & Evaluating Ridge, Lasso, ElasticNet

In [ ]:
models = {
    'Linear (baseline)': LinearRegression(),
    'Ridge (a=1)':       Ridge(alpha=1.0),
    'Lasso (a=1)':       Lasso(alpha=1.0, max_iter=50000),
    'ElasticNet (a=1)':  ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=50000),
}

for name, m in models.items():
    m.fit(X_train_s, y_train)
    tr = r2_score(y_train, m.predict(X_train_s))
    te = r2_score(y_test, m.predict(X_test_s))
    print(f'{name:20s} train R2={tr:.3f}  test R2={te:.3f}')

## 8. Regularization Paths & Choosing Alpha

### Lasso path — watch coefficients shrink to zero as alpha grows

In [ ]:
alphas = np.logspace(-2, 1.5, 40)
coefs = []
for a in alphas:
    l = Lasso(alpha=a, max_iter=50000).fit(X_train_s, y_train)
    coefs.append(l.coef_)
coefs = np.array(coefs)

plt.plot(alphas, coefs)
plt.xscale('log')
plt.xlabel('alpha (log scale)')
plt.ylabel('coefficient value')
plt.title('Lasso regularization path — features drop to 0 as alpha rises')
plt.axhline(0, color='k', lw=0.8)
plt.show()

In [ ]:
# Pick the best alpha automatically via cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100)).fit(X_train_s, y_train)
lasso_cv = LassoCV(alphas=np.logspace(-3, 1.5, 100), max_iter=50000, cv=5).fit(X_train_s, y_train)
enet_cv  = ElasticNetCV(l1_ratio=[.1, .5, .7, .9], alphas=np.logspace(-3, 1.5, 100),
                        max_iter=50000, cv=5).fit(X_train_s, y_train)

print('Best alpha  -> Ridge:', round(ridge_cv.alpha_, 4),
      '| Lasso:', round(lasso_cv.alpha_, 4),
      '| ElasticNet:', round(enet_cv.alpha_, 4), 'l1_ratio:', enet_cv.l1_ratio_)

In [ ]:
# Final comparison with cross-validated alphas
for name, m in [('Ridge CV', ridge_cv), ('Lasso CV', lasso_cv), ('ElasticNet CV', enet_cv)]:
    te = r2_score(y_test, m.predict(X_test_s))
    nonzero = np.sum(m.coef_ != 0)
    print(f'{name:14s} test R2={te:.3f}  features used: {nonzero}/{len(m.coef_)}')

Notice **Lasso uses far fewer features** than Ridge (which keeps all of them) — that's automatic feature selection. All three beat the overfit baseline on the test set.

## 9. Ridge From Scratch (Closed Form)

Ridge has an exact closed-form solution — just the normal equation with $\alpha$ added to the diagonal:

$$\mathbf{w} = (\mathbf{X}^T\mathbf{X} + \alpha \mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

We center the target so we don't penalise the intercept, then compare to scikit-learn.

In [ ]:
alpha = 1.0
n = X_train_s.shape[1]
y_mean = y_train.mean()

# Closed-form Ridge (features already standardized, target centered)
w_scratch = np.linalg.solve(
    X_train_s.T @ X_train_s + alpha * np.eye(n),
    X_train_s.T @ (y_train - y_mean))

ridge_sk = Ridge(alpha=alpha).fit(X_train_s, y_train)

print('Max |coef difference| vs scikit-learn:', np.max(np.abs(w_scratch - ridge_sk.coef_)))
print('Scratch intercept:', round(y_mean, 4), '| sklearn intercept:', round(ridge_sk.intercept_, 4))

## 10. Pros, Cons & When to Use

**Pros**
- Prevents overfitting and stabilises models with many / correlated features.
- Lasso & ElasticNet perform built-in **feature selection** → simpler, more interpretable models.
- Tiny computational cost on top of linear regression.

**Cons**
- Requires choosing `alpha` (and `l1_ratio`) — tune via cross-validation.
- **Features must be scaled**, or the penalty is applied unfairly.
- Still a *linear* model — won't capture genuinely non-linear relationships.
- Lasso can behave erratically when features are highly correlated (ElasticNet is more stable there).

**When to use**
- Whenever you have more features than you trust, or correlated features.
- When you want a sparse, interpretable model (Lasso/ElasticNet).
- As the default upgrade over plain linear regression in real projects.

## 11. Summary

- Regularization adds a penalty on weight size to the loss, trading a little bias for much lower variance.
- **Ridge (L2)** shrinks weights smoothly; **Lasso (L1)** zeros some out (feature selection); **ElasticNet** blends both.
- On the polynomial-expanded Diabetes data, plain linear regression overfit badly, while the regularized models generalised better — and Lasso did it using only a fraction of the features.
- `alpha` is chosen by cross-validation (`RidgeCV` / `LassoCV` / `ElasticNetCV`).
- Ridge's closed-form solution reproduced scikit-learn exactly, confirming we understand the mechanics.